# Deep Learning with PyTorch · nn.Module: layers, parameters and writing models

Macademia · For adults

> **Running in Colab:** Open this file at [colab.research.google.com](https://colab.research.google.com) with **File → Upload notebook**. For a GPU choose **Runtime → Change runtime type → T4 GPU**. Run the cells from top to bottom (Shift + Enter). PyTorch is preinstalled in Colab; to run on your own computer use `pip install torch torchvision`.

In [ ]:
import torch, random, numpy as np
torch.manual_seed(0); random.seed(0); np.random.seed(0)          # reproducibility
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '| device:', device)

## 1. A model class

In [ ]:
from torch import nn

class Siniflandirici(nn.Module):
    def __init__(self, n_in=2, gizli=16, n_out=1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_in, gizli), nn.ReLU(), nn.Linear(gizli, gizli), nn.ReLU(), nn.Linear(gizli, n_out))
    def forward(self, x):
        return self.net(x)

model = Siniflandirici()
print(model)
print(sum(p.numel() for p in model.parameters()), 'parameters')
for name, p in model.named_parameters():
    print(f'{name:14s} {tuple(p.shape)}')

## 2. Tracing shapes layer by layer

In [ ]:
cnn = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                    nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                    nn.Flatten(), nn.Linear(16 * 7 * 7, 10))
x = torch.randn(1, 1, 28, 28)
for layer in cnn:
    x = layer(x)
    n = sum(p.numel() for p in layer.parameters())
    print(f'{layer.__class__.__name__:10s} {str(tuple(x.shape)):18s} {n}')

## 3. The plain-list trap

In [ ]:
class Yanlis(nn.Module):
    def __init__(self):
        super().__init__()
        self.katmanlar = [nn.Linear(4, 4) for _ in range(3)]          # plain list: invisible
class Dogru(nn.Module):
    def __init__(self):
        super().__init__()
        self.katmanlar = nn.ModuleList([nn.Linear(4, 4) for _ in range(3)])
print(len(list(Yanlis().parameters())), len(list(Dogru().parameters())))   # 0 6

## 4. Save and load

In [ ]:
torch.save(model.state_dict(), 'model.pt')
yeni = Siniflandirici()
yeni.load_state_dict(torch.load('model.pt'))
x = torch.randn(3, 2)
print(torch.allclose(model(x), yeni(x)))

## Exercises
1. Compute the parameter count of `nn.Conv2d(3, 16, 3)` by hand first.
2. Remove the second `MaxPool2d` from the CNN above: what must the `Linear` input be?
3. Write a model that finds its input size automatically with `nn.LazyLinear`.